In [1]:
import pandas as pd


In [2]:
imdb = pd.read_csv(
    '../data/title.basics.tsv.gz',
    sep="\t",
    na_values = '\\N', 
    dtype=str,
    )

movies = imdb[imdb['titleType'] == 'movie'][
    ['tconst', 'primaryTitle', 'startYear']   # startYear is needed for "earlier films only"
]


In [3]:
directors = pd.read_csv(
    '../data/title.crew.tsv.gz', sep='\t', na_values='\\N', dtype=str
)[['tconst', 'directors']]

movies = movies.merge(directors, on='tconst', how='left')

In [4]:
movies

,tconst,primaryTitle,startYear,directors
0,tt0000009,Miss Jerry,1894,nm0085156
1,tt0000147,The Corbett-Fitzsimmons Fight,1897,nm0714557
2,tt0000502,Bohemios,1905,nm0063413
3,tt0000574,The Story of the Kelly Gang,1906,nm0846879
4,tt0000591,The Prodigal Son,1907,nm0141150
...,...,...,...,...
760787,tt9916622,Rodolpho Teóphilo - O Legado de um Pioneiro,2015,"nm9272490,nm9272491"
760788,tt9916680,De la ilusión al desconcierto: cine colombiano...,2007,nm0652213
760789,tt9916706,Dankyavar Danka,2013,nm7764440
760790,tt9916730,6 Gunn,2017,nm10538612


In [5]:
movies = movies.rename(columns={
    'tconst': 'imdb_id',
    'primaryTitle': 'title'
})

movies

,imdb_id,title,startYear,directors
0,tt0000009,Miss Jerry,1894,nm0085156
1,tt0000147,The Corbett-Fitzsimmons Fight,1897,nm0714557
2,tt0000502,Bohemios,1905,nm0063413
3,tt0000574,The Story of the Kelly Gang,1906,nm0846879
4,tt0000591,The Prodigal Son,1907,nm0141150
...,...,...,...,...
760787,tt9916622,Rodolpho Teóphilo - O Legado de um Pioneiro,2015,"nm9272490,nm9272491"
760788,tt9916680,De la ilusión al desconcierto: cine colombiano...,2007,nm0652213
760789,tt9916706,Dankyavar Danka,2013,nm7764440
760790,tt9916730,6 Gunn,2017,nm10538612


In [6]:
names = pd.read_csv(
    '../data/name.basics.tsv.gz',
    sep = '\t',
    na_values = '\\N',
    dtype =str )[['nconst', 'primaryName']]

d = movies [['imdb_id', 'directors']].dropna().copy()
d['nconst'] = d['directors'].str.split(',')

d = d.explode('nconst')

d = d.merge(names, on ='nconst', how = 'left')



d['primaryName'] = d['primaryName'].fillna('')
director_names = (
    d.groupby('imdb_id')['primaryName']
    .agg(','.join)
    .rename('director')
    .reset_index()
)

movies = movies.merge(director_names, on='imdb_id', how='left')
movies = movies.drop(columns='directors')

In [7]:
movies


,imdb_id,title,startYear,director
0,tt0000009,Miss Jerry,1894,Alexander Black
1,tt0000147,The Corbett-Fitzsimmons Fight,1897,Enoch J. Rector
2,tt0000502,Bohemios,1905,Ricardo de Baños
3,tt0000574,The Story of the Kelly Gang,1906,Charles Tait
4,tt0000591,The Prodigal Son,1907,Michel Carré
...,...,...,...,...
760787,tt9916622,Rodolpho Teóphilo - O Legado de um Pioneiro,2015,"Angela Gurgel,Ana Célia de Oliveira"
760788,tt9916680,De la ilusión al desconcierto: cine colombiano...,2007,Luis Ospina
760789,tt9916706,Dankyavar Danka,2013,Kanchan Nayak
760790,tt9916730,6 Gunn,2017,Kiran Gawade


In [8]:
movies = movies.dropna(subset=['director'])
movies

,imdb_id,title,startYear,director
0,tt0000009,Miss Jerry,1894,Alexander Black
1,tt0000147,The Corbett-Fitzsimmons Fight,1897,Enoch J. Rector
2,tt0000502,Bohemios,1905,Ricardo de Baños
3,tt0000574,The Story of the Kelly Gang,1906,Charles Tait
4,tt0000591,The Prodigal Son,1907,Michel Carré
...,...,...,...,...
760787,tt9916622,Rodolpho Teóphilo - O Legado de um Pioneiro,2015,"Angela Gurgel,Ana Célia de Oliveira"
760788,tt9916680,De la ilusión al desconcierto: cine colombiano...,2007,Luis Ospina
760789,tt9916706,Dankyavar Danka,2013,Kanchan Nayak
760790,tt9916730,6 Gunn,2017,Kiran Gawade


In [ ]:
imdb_long = d[['imdb_id', 'nconst', 'primaryName']].rename(columns={'primaryName': 'director'})
imdb_long = imdb_long[imdb_long['director'] != '']          # cell 5 filled missing names with ''
# Bring over title and year, movies already has no-director rows dropped
imdb_long = imdb_long.merge(movies[['imdb_id', 'title', 'startYear']], on='imdb_id')
imdb_long = imdb_long.drop_duplicates(subset=['imdb_id', 'nconst'])
imdb_long.to_csv('../data/imdb_movies.csv', index=False)
